# 'Made in Rwanda' Recommender — Evaluation

**Challenge:** S2.T1.3 · AIMS KTT Hackathon  
**Backend:** `paraphrase-multilingual-MiniLM-L12-v2` sentence embeddings  
**Metrics computed:**
- NDCG@5 vs `global_best_match_sku` baseline (overall + per language)
- Local-presence rate — % of queries with ≥1 local product in top-3
- Mean query latency (ms)
- Fairness cap audit — artisan share of top-10 slots across all queries

In [1]:
import math
import time
from collections import defaultdict

import numpy as np
import pandas as pd

from recommender import MadeInRwandaRecommender

## 1 · Load data

In [2]:
catalog   = pd.read_csv("catalog.csv")
queries   = pd.read_csv("queries.csv")
click_log = pd.read_csv("click_log.csv")

print(f"Catalog  : {len(catalog):>4} products  |  {catalog['category'].nunique()} categories  |  {catalog['artisan_id'].nunique()} artisans")
print(f"Queries  : {len(queries):>4} queries   |  language breakdown:")
print(queries['language'].value_counts().to_string())
print(f"Click log: {len(click_log):>4} events")

Catalog  :  400 products  |  5 categories  |  80 artisans
Queries  :  120 queries   |  language breakdown:
language
en               56
fr               34
code_switched    17
misspelled       13
Click log: 5000 events


## 2 · Initialise recommender

In [3]:
rec = MadeInRwandaRecommender()
print("Recommender ready.")

Recommender ready.


## 3 · Run all 120 queries and collect results

In [4]:
N_EVAL = 5   # rank cutoff for NDCG
N_LOCAL = 3  # rank cutoff for local-presence rate
N_FAIR  = 10 # rank cutoff for fairness cap audit

rows = []
for _, q in queries.iterrows():
    t0 = time.perf_counter()
    recs = rec.recommend(q["query_text"], n=N_FAIR)  # fetch top-10 for fairness audit
    elapsed_ms = (time.perf_counter() - t0) * 1_000

    top_skus = recs["sku"].tolist()
    rows.append({
        "query_id"            : q["query_id"],
        "query_text"          : q["query_text"],
        "language"            : q["language"],
        "global_best_sku"     : q["global_best_match_sku"],
        "top_skus"            : top_skus,
        "top_artisans"        : recs["artisan_id"].tolist(),
        "elapsed_ms"          : elapsed_ms,
        "fallback_injected"   : recs["fallback_injected"].any(),
    })

results = pd.DataFrame(rows)
print(f"Evaluated {len(results)} queries.  Mean latency: {results['elapsed_ms'].mean():.1f} ms")

Evaluated 120 queries.  Mean latency: 8.4 ms


## 4 · NDCG@5

Binary relevance: the `global_best_match_sku` is the single relevant document per query.  
NDCG@5 per query = `1 / log2(rank + 1)` if that SKU appears in positions 1–5, else 0.  
IDCG@5 = 1 (ideal: relevant doc at rank 1).

### Why NDCG@5 is expected to be low here

The catalog contains ~5 products per artisan across 5 categories. Many products share
nearly identical embeddings (e.g. all "cow leather boots" cluster tightly). The
`global_best_match_sku` is one specific SKU the global algorithm picks — but our
recommender intentionally:

1. **Diversifies across artisans** via the fairness cap (max 1 slot per artisan in top-10),
   which displaces the global-best SKU when an equally-similar product from a different
   artisan ranks ahead of it.
2. **Blends popularity** (10% weight), shifting tie-breaks toward higher-click products
   that may not be the global-best SKU.

A low NDCG@5 relative to the global baseline is therefore the **expected, accepted
tradeoff** of a local-first, fairness-capped recommender. The primary success
metric is **local-presence rate**.

In [5]:
def ndcg_at_k(retrieved_skus: list, relevant_sku: str, k: int = 5) -> float:
    """Binary-relevance NDCG@k with a single relevant document."""
    for rank, sku in enumerate(retrieved_skus[:k], start=1):
        if sku == relevant_sku:
            return 1.0 / math.log2(rank + 1)
    return 0.0

results["ndcg_at_5"] = results.apply(
    lambda r: ndcg_at_k(r["top_skus"], r["global_best_sku"], k=N_EVAL), axis=1
)

overall_ndcg = results["ndcg_at_5"].mean()
print(f"NDCG@5 (overall, n={len(results)}): {overall_ndcg:.4f}")

NDCG@5 (overall, n=120): 0.0119


In [6]:
ndcg_by_lang = (
    results.groupby("language")["ndcg_at_5"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "NDCG@5", "count": "n_queries"})
    .sort_values("NDCG@5", ascending=False)
)
ndcg_by_lang["NDCG@5"] = ndcg_by_lang["NDCG@5"].map("{:.4f}".format)
print("NDCG@5 by language:")
print(ndcg_by_lang.to_string())

NDCG@5 by language:
               NDCG@5  n_queries
language                        
misspelled     0.0331         13
fr             0.0294         34
en             0.0000         56
code_switched  0.0000         17


## 5 · Local-presence rate

Percentage of queries where ≥1 product in the top-3 results is a Made-in-Rwanda product.

Since our catalog contains *only* Made-in-Rwanda products, every returned result is
local by construction. The local-boost rule additionally guarantees a curated local
result at rank 1 even for queries with zero semantic similarity — so the rate is 100%.

In [7]:
local_skus = set(catalog["sku"])  # all catalog SKUs are Made in Rwanda

def local_presence(top_skus: list, local_set: set, k: int = 3) -> bool:
    return any(s in local_set for s in top_skus[:k])

results["has_local_top3"] = results["top_skus"].apply(
    lambda skus: local_presence(skus, local_skus, k=N_LOCAL)
)

local_rate = results["has_local_top3"].mean() * 100
fallback_rate = results["fallback_injected"].mean() * 100

print(f"Local-presence rate (top-{N_LOCAL}): {local_rate:.1f}%")
print(f"Queries where local-boost injected a fallback: {fallback_rate:.1f}%")

Local-presence rate (top-3): 100.0%
Queries where local-boost injected a fallback: 0.0%


In [8]:
local_by_lang = (
    results.groupby("language")["has_local_top3"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "local_presence_rate", "count": "n_queries"})
)
local_by_lang["local_presence_rate"] = (local_by_lang["local_presence_rate"] * 100).map("{:.1f}%".format)
print("Local-presence rate by language:")
print(local_by_lang.to_string())

Local-presence rate by language:
              local_presence_rate  n_queries
language                                    
code_switched              100.0%         17
en                         100.0%         56
fr                         100.0%         34
misspelled                 100.0%         13


## 6 · Query latency

In [9]:
lat = results["elapsed_ms"]
limit_ok = lat.max() < 250
print(f"Latency (ms) across {len(lat)} queries:")
print(f"  mean   : {lat.mean():.1f}")
print(f"  median : {lat.median():.1f}")
print(f"  p95    : {lat.quantile(0.95):.1f}")
print(f"  max    : {lat.max():.1f}")
print(f"  limit  : 250.0  ->  {'PASS' if limit_ok else 'FAIL'}")

Latency (ms) across 120 queries:
  mean   : 8.4
  median : 8.0
  p95    : 9.4
  max    : 45.0
  limit  : 250.0  ->  PASS


## 7 · Fairness cap audit (stretch goal)

**Requirement:** no single artisan occupies more than 15% of the top-10 recommendations
for any given query (i.e. ≤1 slot out of 10 per artisan per query).  
The cap is enforced inside `recommend()` via `_apply_fairness_cap()`.  
Here we verify empirically that the constraint is never violated.

In [10]:
CAP = 0.15  # 15% of top-10 = 1.5, floor -> max 1 slot per artisan per query
max_slots = max(1, int(N_FAIR * CAP))

violations = []
for _, row in results.iterrows():
    top_artisans = row["top_artisans"][:N_FAIR]
    from collections import Counter
    counts = Counter(top_artisans)
    for artisan, cnt in counts.items():
        frac = cnt / N_FAIR
        if frac > CAP:
            violations.append({
                "query_id": row["query_id"],
                "artisan_id": artisan,
                "slots": cnt,
                "fraction": frac,
            })

if violations:
    print(f"VIOLATIONS ({len(violations)}):")
    print(pd.DataFrame(violations).to_string())
else:
    print(f"Fairness cap holds: no artisan exceeded {CAP*100:.0f}% of top-{N_FAIR} in any query. [PASS]")

Fairness cap holds: no artisan exceeded 15% of top-10 in any query. [PASS]


In [11]:
# Aggregate artisan appearance frequency across all queries (top-10)
all_artisans = [a for row in results["top_artisans"] for a in row[:N_FAIR]]
artisan_counts = pd.Series(all_artisans).value_counts()
total_slots = len(results) * N_FAIR

artisan_share = (artisan_counts / total_slots * 100).rename("share_%")
print(f"Top-10 most-appearing artisans across all {len(results)} queries (total slots: {total_slots}):")
print(artisan_share.head(10).map("{:.2f}%".format).to_string())
print(f"\nMax share: {artisan_share.max():.2f}%  (global cap threshold: {CAP*100:.0f}% per query)")

Top-10 most-appearing artisans across all 120 queries (total slots: 1200):
ART-057    3.25%
ART-011    3.00%
ART-052    2.92%
ART-035    2.50%
ART-030    2.50%
ART-078    2.25%
ART-060    2.25%
ART-047    2.25%
ART-063    2.25%
ART-068    2.25%

Max share: 3.25%  (global cap threshold: 15% per query)


## 8 · Summary

In [12]:
summary = pd.DataFrame([
    {"Metric": "NDCG@5 (vs global baseline)",         "Value": f"{overall_ndcg:.4f}"},
    {"Metric": f"Local-presence rate (top-{N_LOCAL})", "Value": f"{local_rate:.1f}%"},
    {"Metric": "Mean query latency",                   "Value": f"{lat.mean():.1f} ms"},
    {"Metric": "p95 query latency",                    "Value": f"{lat.quantile(0.95):.1f} ms"},
    {"Metric": "Max query latency",                    "Value": f"{lat.max():.1f} ms"},
    {"Metric": "250 ms constraint",                    "Value": "PASS" if lat.max() < 250 else "FAIL"},
    {"Metric": "Fairness cap violations",              "Value": str(len(violations))},
    {"Metric": "Queries with fallback injected",       "Value": f"{fallback_rate:.1f}%"},
]).set_index("Metric")

print(summary.to_string())

                                  Value
Metric                                 
NDCG@5 (vs global baseline)      0.0119
Local-presence rate (top-3)      100.0%
Mean query latency               8.4 ms
p95 query latency                9.4 ms
Max query latency               45.0 ms
250 ms constraint                  PASS
Fairness cap violations               0
Queries with fallback injected     0.0%
